# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 1.1 Load Dataset

This section loads the provided dataset files from the `data` folder, including the evidence corpus, training claims, and development claims. We also print the number of evidence passages and claims to verify that the files are loaded correctly.

In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("data")

with open(DATA_DIR / "evidence.json", "r") as f:
    evidence = json.load(f)

with open(DATA_DIR / "train-claims.json", "r") as f:
    train_claims = json.load(f)

with open(DATA_DIR / "dev-claims.json", "r") as f:
    dev_claims = json.load(f)

print("Evidence:", len(evidence))
print("Train claims:", len(train_claims))
print("Dev claims:", len(dev_claims))

first_eid = list(evidence.keys())[0]
print(first_eid, evidence[first_eid])

first_cid = list(dev_claims.keys())[0]
print(first_cid, dev_claims[first_cid])

Evidence: 1208827
Train claims: 1228
Dev claims: 154
evidence-0 John Bennet Lawes, English entrepreneur and agricultural scientist
claim-752 {'claim_text': '[South Australia] has the most expensive electricity in the world.', 'claim_label': 'SUPPORTS', 'evidences': ['evidence-67732', 'evidence-572512']}


## 1.2 Different Evidence Retrieval

This section implements and compares different evidence retrieval methods. We first build sparse lexical retrievers, including TF-IDF and BM25. We then add a dense retrieval component to reduce lexical mismatch between claims and evidence passages. Finally, we combine multiple retrieval rankings using Reciprocal Rank Fusion (RRF).

The purpose of this section is to retrieve relevant evidence IDs for each claim before the claim classification stage.

## 1.2.1 Build TF-IDF Evidence Index

This section builds a TF-IDF index over all evidence passages. Each evidence text is converted into a sparse vector representation. The retriever later compares each claim vector against this evidence matrix to find the most relevant evidence passages.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

#把evidence转成list
evidence_ids = list(evidence.keys())
evidence_texts = [evidence[eid] for eid in evidence_ids]

print("Total evidence:", len(evidence_texts))
print("Example:", evidence_texts[0])

# TF-IDF
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=None,
    ngram_range=(1, 2),
    max_features=500000,
    sublinear_tf=True,
    min_df=2
)

evidence_matrix = vectorizer.fit_transform(evidence_texts)

print("TF-IDF matrix shape:", evidence_matrix.shape)

Total evidence: 1208827
Example: John Bennet Lawes, English entrepreneur and agricultural scientist
TF-IDF matrix shape: (1208827, 500000)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve_tfidf_top_k(claim_text, k=5):
    claim_vec = vectorizer.transform([claim_text])
    scores = cosine_similarity(claim_vec, evidence_matrix).flatten()

    top_indices = np.argsort(scores)[::-1][:k]

    return [evidence_ids[i] for i in top_indices]

In [ ]:
# test一个claim
claim_id = list(dev_claims.keys())[0]
claim_text = dev_claims[claim_id]["claim_text"]

print("Claim:", claim_text)

retrieved = retrieve_tfidf_top_k(claim_text, k=5)

print("Retrieved evidence IDs:", retrieved)
print("Gold evidence:", dev_claims[claim_id]["evidences"])

Claim: [South Australia] has the most expensive electricity in the world.
Retrieved evidence IDs: ['evidence-67732', 'evidence-572512', 'evidence-509525', 'evidence-32494', 'evidence-147175']
Gold evidence: ['evidence-67732', 'evidence-572512']


## 1.2.2 BM25 Evidence Retrieval

This section implements a BM25 retriever as an alternative sparse retrieval baseline. BM25 is a strong lexical retrieval method that ranks evidence passages based on term matching while considering document length normalisation and term saturation.

In [ ]:
# If rank_bm25 is not installed, run this once:
!pip install rank-bm25

In [ ]:
import re
from rank_bm25 import BM25Okapi
from tqdm import tqdm

def bm25_tokenize(text):
    text = text.lower()
    text = text.replace("[", " ").replace("]", " ")
    tokens = re.findall(r"[a-z0-9]+", text)
    return tokens

# Tokenise all evidence passages
tokenized_evidence = [
    bm25_tokenize(text) 
    for text in tqdm(evidence_texts, desc="Tokenising evidence")
]

bm25 = BM25Okapi(tokenized_evidence)

print("BM25 index built.")
print("Number of evidence passages:", len(tokenized_evidence))

Tokenising evidence: 100%|██████████| 1208827/1208827 [00:06<00:00, 194978.10it/s]


BM25 index built.
Number of evidence passages: 1208827


In [ ]:
def retrieve_bm25_top_k(claim_text, k=5):
    """ 
        Retrieve top-k evidence IDs for a claim using BM25. 
        Args:
            claim_text (str): The claim text to retrieve evidence for.
            k (int): The number of top evidence passages to retrieve for the claim.
        Returns:
            List[str]: A list of evidence IDs corresponding to the top-k retrieved passages.
    """
    query_tokens = bm25_tokenize(claim_text)
    scores = bm25.get_scores(query_tokens)

    top_indices = np.argpartition(scores, -k)[-k:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]

    return [evidence_ids[i] for i in top_indices]

In [ ]:
bm25_retrieved = retrieve_bm25_top_k(claim_text, k=5)

print("Claim:", claim_text)
print("BM25 retrieved evidence IDs:", bm25_retrieved)
print("Gold evidence:", dev_claims[claim_id]["evidences"])

Claim: [South Australia] has the most expensive electricity in the world.
BM25 retrieved evidence IDs: ['evidence-67732', 'evidence-572512', 'evidence-147175', 'evidence-250017', 'evidence-65625']
Gold evidence: ['evidence-67732', 'evidence-572512']


In [ ]:
def evaluate_retrieval_overlap(claims, retriever_fn, k=5):
    """
    Evaluate how often retrieved top-k evidence contains gold evidence。
    Args:
        claims (dict): A dictionary of claims, where each key is a claim ID and the value is a dictionary containing "claim_text" and "evidences".
        retriever_fn (function): A function that takes a claim text and returns a list of retrieved evidence IDs.
        k (int): The number of top evidence passages to retrieve for each claim.
    Returns:
        dict: A summary of retrieval performance, including hit@k, average recall@k, and average overlap.
        list: A list of detailed results for each claim, including retrieved evidence and overlap with gold evidence.
    """
    total_claims = len(claims)
    hit_count = 0
    total_recall = 0.0
    total_overlap = 0

    results = []

    for claim_id, claim_data in claims.items():
        claim_text = claim_data["claim_text"]
        gold_ids = set(claim_data["evidences"])

        retrieved_ids = retriever_fn(claim_text, k=k)
        retrieved_set = set(retrieved_ids)

        overlap = gold_ids & retrieved_set

        if len(overlap) > 0:
            hit_count += 1

        recall = len(overlap) / len(gold_ids) if len(gold_ids) > 0 else 0.0

        total_recall += recall
        total_overlap += len(overlap)

        results.append({
            "claim_id": claim_id,
            "claim_text": claim_text,
            "gold_evidence": list(gold_ids),
            "retrieved_evidence": retrieved_ids,
            "overlap": list(overlap),
            "hit": len(overlap) > 0,
            "recall": recall
        })

    hit_at_k = hit_count / total_claims
    avg_recall_at_k = total_recall / total_claims
    avg_overlap = total_overlap / total_claims

    summary = {
        "k": k,
        "total_claims": total_claims,
        "hit_count": hit_count,
        "hit_at_k": hit_at_k,
        "avg_recall_at_k": avg_recall_at_k,
        "avg_overlap": avg_overlap
    }

    return summary, results

In [ ]:
for k in [5, 10, 20, 50, 100]:
    summary, _ = evaluate_retrieval_overlap(
        dev_claims,
        retriever_fn=retrieve_tfidf_top_k,
        k=k
    )
    print(summary)

{'k': 5, 'total_claims': 154, 'hit_count': 29, 'hit_at_k': 0.18831168831168832, 'avg_recall_at_k': 0.09415584415584415, 'avg_overlap': 0.22077922077922077}
{'k': 10, 'total_claims': 154, 'hit_count': 39, 'hit_at_k': 0.2532467532467532, 'avg_recall_at_k': 0.12759740259740254, 'avg_overlap': 0.3246753246753247}
{'k': 20, 'total_claims': 154, 'hit_count': 55, 'hit_at_k': 0.35714285714285715, 'avg_recall_at_k': 0.1708874458874458, 'avg_overlap': 0.474025974025974}
{'k': 50, 'total_claims': 154, 'hit_count': 75, 'hit_at_k': 0.487012987012987, 'avg_recall_at_k': 0.24047619047619045, 'avg_overlap': 0.7142857142857143}
{'k': 100, 'total_claims': 154, 'hit_count': 93, 'hit_at_k': 0.6038961038961039, 'avg_recall_at_k': 0.3391774891774894, 'avg_overlap': 1.0}


In [ ]:
summary_tfidf, results_tfidf = evaluate_retrieval_overlap(
    dev_claims,
    retriever_fn=retrieve_tfidf_top_k,
    k=5
)

summary_tfidf

{'k': 5,
 'total_claims': 154,
 'hit_count': 29,
 'hit_at_k': 0.18831168831168832,
 'avg_recall_at_k': 0.09415584415584415,
 'avg_overlap': 0.22077922077922077}

In [ ]:
# for k in [5, 10, 20, 50, 100]:
#     summary, _ = evaluate_retrieval_overlap(
#         dev_claims,
#         retriever_fn=retrieve_bm25_top_k,
#         k=k
#     )
#     print(summary)

In [45]:
summary_bm25, results_bm25 = evaluate_retrieval_overlap(
    dev_claims,
    retriever_fn=retrieve_bm25_top_k,
    k=5
)

summary_bm25

{'k': 5,
 'total_claims': 154,
 'hit_count': 50,
 'hit_at_k': 0.3246753246753247,
 'avg_recall_at_k': 0.15627705627705624,
 'avg_overlap': 0.44155844155844154}

## 1.2.3 Dense Retrieval over Sparse Candidates

Dense retrieval is used to reduce lexical mismatch between claims and evidence passages. Instead of encoding the entire evidence corpus, we first retrieve a candidate pool using a sparse retriever (TF-IDF or BM25), then rerank those candidates with sentence embeddings. Standalone pipelines use either TF-IDF or BM25 as the candidate source; in the RRF setup (Section 1.2.4), dense reranking is applied separately to each sparse list. This keeps dense retrieval computationally feasible.

In [ ]:
# If sentence-transformers is not installed, run this once:
!pip install sentence-transformers

We used the open-source sentence-transformers/all-MiniLM-L6-v2 model, which is released under the Apache-2.0 license, as a lightweight dense embedding model for candidate reranking. https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

In [ ]:
from sentence_transformers import SentenceTransformer, util

dense_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
def dense_rank_candidates(claim_text, candidate_ids, batch_size=64):
    """
    Rerank candidate evidence IDs using sentence embedding similarity.
    Args:
        claim_text (str): The claim text to rank evidence for.
        candidate_ids (List[str]): A list of candidate evidence IDs to rerank.
        batch_size (int): The batch size to use when encoding candidate texts.
    Returns:
        List[str]: A list of evidence IDs sorted by relevance to the claim, with the most relevant first.
    """
    if len(candidate_ids) == 0:
        return []

    candidate_texts = [evidence[eid] for eid in candidate_ids]

    claim_emb = dense_model.encode(
        claim_text,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    candidate_embs = dense_model.encode(
        candidate_texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    scores = util.cos_sim(claim_emb, candidate_embs)[0].cpu().numpy()
    ranked_indices = np.argsort(scores)[::-1]

    return [candidate_ids[i] for i in ranked_indices]

In [ ]:
# gold evidence is often retrieved by TF-IDF within the top 100
def retrieve_tfidf_dense_top_k(claim_text, k=5, candidate_k=100):
    """
    Retrieve top-k evidence IDs using TF-IDF candidates followed by dense reranking.
    Args:
        claim_text (str): The claim text to retrieve evidence for.
        k (int): The number of top evidence passages to return after reranking.
        candidate_k (int): The number of top candidates to retrieve using TF-IDF before reranking.
    Returns:
        List[str]: A list of evidence IDs corresponding to the top-k retrieved passages after reranking.
    """
    tfidf_candidates = retrieve_tfidf_top_k(claim_text, k=candidate_k)

    dense_ranked = dense_rank_candidates(
        claim_text=claim_text,
        candidate_ids=tfidf_candidates
    )

    return dense_ranked[:k]

In [49]:
summary_tfidf_dense, results_tfidf_dense = evaluate_retrieval_overlap(
    dev_claims,
    retriever_fn=lambda claim_text, k=5: retrieve_tfidf_dense_top_k(
        claim_text,
        k=k,
        candidate_k=100
    ),
    k=5
)

summary_tfidf_dense

{'k': 5,
 'total_claims': 154,
 'hit_count': 69,
 'hit_at_k': 0.44805194805194803,
 'avg_recall_at_k': 0.22099567099567094,
 'avg_overlap': 0.5909090909090909}

In [ ]:
def retrieve_bm25_dense_top_k(claim_text, k=5, candidate_k=100):
    """
    Retrieve top-k evidence IDs using BM25 candidates followed by dense reranking.
    """
    bm25_candidates = retrieve_bm25_top_k(claim_text, k=candidate_k)

    dense_ranked = dense_rank_candidates(
        claim_text=claim_text,
        candidate_ids=bm25_candidates
    )

    return dense_ranked[:k]

In [50]:
summary_bm25_dense, results_bm25_dense = evaluate_retrieval_overlap(
    dev_claims,
    retriever_fn=lambda claim_text, k=5: retrieve_bm25_dense_top_k(
        claim_text,
        k=k,
        candidate_k=100
    ),
    k=5
)

summary_bm25_dense

{'k': 5,
 'total_claims': 154,
 'hit_count': 69,
 'hit_at_k': 0.44805194805194803,
 'avg_recall_at_k': 0.2160173160173159,
 'avg_overlap': 0.6103896103896104}

## 1.2.4 Reciprocal Rank Fusion

TF-IDF, BM25, and dense retrieval capture different relevance signals. TF-IDF and BM25 rely on lexical matching, while dense retrieval captures semantic similarity. Dense reranking is computed twice: once over the TF-IDF candidate list and once over the BM25 candidate list, so each sparse retriever is paired with its own dense ordering. We fuse these four ranked lists using Reciprocal Rank Fusion (RRF), which gives higher scores to evidence passages that are ranked highly by multiple retrievers.

In [ ]:
from collections import defaultdict

def reciprocal_rank_fusion(rank_lists, top_k=5, rrf_k=60):
    """
    Combine multiple ranked evidence lists using Reciprocal Rank Fusion.
    Args:
        rank_lists (List[List[str]]): A list of ranked evidence ID lists from different retrieval methods.
        top_k (int): The number of top evidence IDs to return after fusion.
        rrf_k (int): The RRF parameter that controls the influence of lower-ranked items.
    Returns:
        List[str]: A list of evidence IDs corresponding to the top-k fused rankings.
    """
    scores = defaultdict(float)

    for rank_list in rank_lists:
        for rank, eid in enumerate(rank_list):
            scores[eid] += 1.0 / (rrf_k + rank + 1)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [eid for eid, score in ranked[:top_k]]

In [ ]:
def retrieve_rrf_top_k(claim_text, k=5, candidate_k=100, rrf_k=60):
    """
    Retrieve top-k evidence IDs using RRF over TF-IDF, BM25, and two dense rankings.

        Args:
            claim_text (str): The claim text to retrieve evidence for.
            k (int): The number of top evidence passages to return after fusion.
            candidate_k (int): The number of top candidates to retrieve using each method before fusion.
            rrf_k (int): The RRF parameter that controls the influence of lower-ranked items.
        returns:
            List[str]: A list of evidence IDs corresponding to the top-k retrieved passages after fusion.
    """
    tfidf_rank = retrieve_tfidf_top_k(claim_text, k=candidate_k)
    bm25_rank = retrieve_bm25_top_k(claim_text, k=candidate_k)

    dense_tfidf = dense_rank_candidates(claim_text, tfidf_rank)
    dense_bm25 = dense_rank_candidates(claim_text, bm25_rank)

    return reciprocal_rank_fusion(
        rank_lists=[tfidf_rank, bm25_rank, dense_tfidf, dense_bm25],
        top_k=k,
        rrf_k=rrf_k
    )

In [ ]:
summary_rrf, results_rrf = evaluate_retrieval_overlap(
    dev_claims,
    retriever_fn=lambda claim_text, k=5: retrieve_rrf_top_k(
        claim_text,
        k=k,
        candidate_k=100
    ),
    k=5
)

summary_rrf

RRF dev evaluation wall time: 506.80s (8.45 min)


{'k': 5,
 'total_claims': 154,
 'hit_count': 61,
 'hit_at_k': 0.3961038961038961,
 'avg_recall_at_k': 0.19740259740259733,
 'avg_overlap': 0.5194805194805194}

## 1.2.5 Generate Retrieval Prediction Files

This section generates development prediction files for different retrieval methods: TF-IDF, BM25, TF-IDF + dense, BM25 + dense, and RRF.

In [57]:
import json

def build_retrieval_predictions(claims, retriever_fn, k=5):
    """
    Build prediction JSON for retrieval evaluation.
    """
    predictions = {}

    for claim_id, claim_data in tqdm(claims.items(), desc="Retrieving evidence"):
        claim_text = claim_data["claim_text"]
        retrieved_ids = retriever_fn(claim_text, k=k)

        predictions[claim_id] = {
            "claim_label": "SUPPORTS",  # placeholder label
            "evidences": retrieved_ids
        }

    return predictions

In [58]:
# TF-IDF predictions
dev_pred_tfidf = build_retrieval_predictions(
    dev_claims,
    retriever_fn=retrieve_tfidf_top_k,
    k=5
)

with open("dev_predictions_tfidf_k5.json", "w") as f:
    json.dump(dev_pred_tfidf, f, indent=2)

Retrieving evidence: 100%|██████████| 154/154 [01:04<00:00,  2.41it/s]


In [59]:
# BM25 predictions
dev_pred_bm25 = build_retrieval_predictions(
    dev_claims,
    retriever_fn=retrieve_bm25_top_k,
    k=5
)

with open("dev_predictions_bm25_k5.json", "w") as f:
    json.dump(dev_pred_bm25, f, indent=2)

Retrieving evidence: 100%|██████████| 154/154 [06:40<00:00,  2.60s/it]


In [62]:
# TF-IDF sparse candidates + dense rerank
def dense_retriever_wrapper(claim_text, k=5):
    return retrieve_tfidf_dense_top_k(claim_text, k=k, candidate_k=100)

dev_pred_dense = build_retrieval_predictions(
    dev_claims,
    retriever_fn=dense_retriever_wrapper,
    k=5
)

with open("dev_predictions_tfidf_dense_k5.json", "w") as f:
    json.dump(dev_pred_dense, f, indent=2)

Retrieving evidence: 100%|██████████| 154/154 [01:14<00:00,  2.07it/s]


In [61]:
# BM25 sparse candidates + dense rerank
def bm25_dense_retriever_wrapper(claim_text, k=5):
    return retrieve_bm25_dense_top_k(claim_text, k=k, candidate_k=100)

dev_pred_bm25_dense = build_retrieval_predictions(
    dev_claims,
    retriever_fn=bm25_dense_retriever_wrapper,
    k=5
)

with open("dev_predictions_bm25_dense_k5.json", "w") as f:
    json.dump(dev_pred_bm25_dense, f, indent=2)

Retrieving evidence: 100%|██████████| 154/154 [07:05<00:00,  2.76s/it]


# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*